# Quick Test — Atlas Trade Prompt Smoke Test

Run this notebook **after** `setup.ipynb` has loaded the model.  
Purpose: verify that the system prompt, message builder, and `chat()` function all work correctly before a real session.

This notebook does **not** reload the model — it assumes `model`, `tokenizer`, and `chat()` are already in scope from `setup.ipynb`.

## 1 — Verify imports and prompt content

In [ ]:
import sys, pathlib

REPO_DIR = "/content/unsloth-qwen35-trading-assistant"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.trading_prompt import (
    PROJECT_NAME,
    DEFAULT_MODEL_NAME,
    DEFAULT_MAX_SEQ_LENGTH,
    TRADING_SYSTEM_PROMPT,
    build_trading_messages,
)

print(f"Project         : {PROJECT_NAME}")
print(f"Default model   : {DEFAULT_MODEL_NAME}")
print(f"Default context : {DEFAULT_MAX_SEQ_LENGTH} tokens")
print(f"Prompt length   : {len(TRADING_SYSTEM_PROMPT)} chars")
print()
print("--- System prompt preview (first 300 chars) ---")
print(TRADING_SYSTEM_PROMPT[:300], "...")

## 2 — Test build_trading_messages() output

In [ ]:
# Single turn — no history
msgs = build_trading_messages(
    user_message="ETH/USDT 1H breakout setup değerlendirmesi",
    market_context="Fiyat: 3200, RSI: 62, EMA20 üzerinde",
)

print(f"Message count : {len(msgs)}")
for m in msgs:
    preview = m['content'][:80].replace('\n', ' ')
    print(f"  [{m['role']:9}] {preview}...")

print()

# Multi-turn — with history
fake_history = [
    {"role": "user",      "content": "BTC nasıl görünüyor?"},
    {"role": "assistant", "content": "Yükseliş trendi devam ediyor..."},
]
msgs_with_history = build_trading_messages(
    user_message="Peki stop nereye koymalıyım?",
    history=fake_history,
)

print(f"Message count with history: {len(msgs_with_history)}")
for m in msgs_with_history:
    preview = m['content'][:60].replace('\n', ' ')
    print(f"  [{m['role']:9}] {preview}...")

## 3 — Live inference smoke test

Sends a minimal message to the model and checks that the response is non-empty and coherent.  
Expected: a brief response explaining that more data is needed, or a conditional framework.

In [ ]:
# Minimal input — Atlas Trade should ask for more context or give a conditional view
response = chat(
    user_message="SOL/USDT analiz et",
    reset=True,
    max_new_tokens=512,
)

print("Response length (chars):", len(response))
print()
print(response)

## 4 — Trade-audit mode test

Uses a Turkish trigger phrase to activate trade-audit mode. Expected: structured 6-part evaluation.

In [ ]:
response = chat(
    user_message="Bu setup nasıl, entry uygun mu?",
    market_context="""
Sembol    : BTC/USDT
Timeframe : 1H
Fiyat     : 84,200 USDT
Durum     : Yatay konsolidasyon sonrası 84,000 direncini kırdı
RSI(14)   : 58 — nötr, aşırı alım yok
Hacim     : kırılış mumunda belirgin artış
EMA20     : 83,400 (fiyat altında, destek)
Stop fikri: 83,200 altı
Hedef     : 86,000
""",
    reset=True,
    max_new_tokens=1024,
)

print(response)

## 5 — Multi-turn continuity test

Checks that the model remembers the previous trade context when answering a follow-up question.

In [ ]:
# Follow-up — should reference the BTC setup from cell 4 automatically
response = chat(
    user_message="Tamam, hedefleri ikiye bölelim. TP1 ve TP2 için mantıklı bölgeler neler olur?",
    max_new_tokens=512,
)
print(response)

## 6 — Thinking mode test (Qwen3 chain-of-thought)

Sends a complex question with `thinking=True`. Expected: longer, more detailed reasoning visible in the output.

In [ ]:
response = chat(
    user_message="BTC/USDT için haftalık kapanış seviyesi bu haftaki pozisyon yönetimini nasıl etkiler?",
    market_context="BTC haftalık kapanış: 84,000 üzerinde. Önceki haftalık direnç: 85,500.",
    reset=True,
    thinking=True,
    max_new_tokens=1500,
)
print(response)

---

## Test summary

| Test | What it checks |
|------|----------------|
| 1    | Imports and prompt content |
| 2    | `build_trading_messages()` single and multi-turn |
| 3    | Minimal input handling |
| 4    | Trade-audit mode via Turkish trigger |
| 5    | Conversation history continuity |
| 6    | Thinking mode (`/think`) |

If all cells run without error and produce coherent responses, the setup is working correctly.